# whisper-repro on Colab

COMP8240 — reproducing Whisper's evaluation.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Results are written to Google Drive, so a dropped session costs nothing —
every stage resumes from what is already on disk. Just re-run the cells.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())

## 2. Mount Drive

Everything the pipeline produces lands here, so it survives the session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code

Two options — uncomment whichever applies.

Once the repo is on GitHub, the clone is the cleaner path: `git pull` picks
up changes you make locally.

In [ ]:
# --- Option A: clone from GitHub (once you've pushed) ---
# !git clone https://github.com/<you>/whisper-repro.git /content/whisper-repro

# --- Option B: upload whisper-repro-scaffold.tar.gz via the Files panel ---
# !tar -xzf /content/whisper-repro-scaffold.tar.gz -C /content/

%cd /content/whisper-repro
!pip install -e . -q
print('installed')

ffmpeg is already on Colab, and torch is preinstalled with CUDA — pip keeps
the existing build rather than pulling a CPU wheel.

Note the adapter reads LibriSpeech with `Audio(decode=False)` and decodes via
soundfile, so the `torchcodec` breakage that bites on local installs never
comes up here.

## 4. Point the data root at Drive

`%env` matters: the path is read at import time, and `!` commands run in a
subprocess, so this has to be set before anything else runs.

**Trade-off worth knowing:** Drive is slow for many small files. For a few
hundred utterances it's fine. If you scale to thousands, switch the data root
to `/content/whisper-data` (local, fast) and copy results to Drive at the end
— the last cell does that.

In [ ]:
%env WHISPER_REPRO_DATA=/content/drive/MyDrive/whisper-repro-data
!mkdir -p "$WHISPER_REPRO_DATA"
!whisper-repro datasets

## 5. Reproduce the paper's LibriSpeech number

`run` = prepare + transcribe + score. Start with `base` to confirm the path
works end to end, then step up to `large-v2` for the headline figure.

In [ ]:
!whisper-repro run --dataset librispeech_test_clean --model base --limit 50

In [ ]:
# The reproduction target: large-v2, paper reports 2.7% on test-clean.
# Audio is already prepared, so this only runs the transcribe + score stages.
!whisper-repro run --dataset librispeech_test_clean --model large-v2 --limit 50

## 6. Find examples worth writing about

Sorts by error rate and prints reference/hypothesis pairs — this is where the
worked example in the proposal came from.

In [ ]:
!whisper-repro inspect --dataset librispeech_test_clean --model large-v2 --worst 10

## 7. Your constructed Nepali dataset

Put the recordings in Drive:

```
MyDrive/nepali-recordings/
  transcripts.csv      filename,reference[,language,speaker_id,...]
  audio/               any format — converted to 16 kHz mono FLAC on the way in
```

`--source-dir` overrides the config file, so no YAML is needed on Colab.

In [ ]:
SOURCE = '/content/drive/MyDrive/nepali-recordings'

!whisper-repro prepare --dataset nepali_custom --source-dir "$SOURCE"
!whisper-repro transcribe --dataset nepali_custom --model large-v2
!whisper-repro score --dataset nepali_custom --model large-v2

The Basic normaliser is selected automatically for Devanagari, and CER is
reported alongside WER — word-level rates are unreliable for Nepali.

In [ ]:
# Does Whisper even identify the language correctly? A separate question
# from recognition accuracy, and an interesting result either way.
!whisper-repro transcribe --dataset nepali_custom --model large-v2 --detect-language --force
!whisper-repro score --dataset nepali_custom --model large-v2

## 8. Results

Summaries and per-utterance CSVs. If you used a local data root, copy to Drive
here.

In [ ]:
import glob, json, os
root = os.environ['WHISPER_REPRO_DATA']
for f in sorted(glob.glob(f'{root}/results/*.summary.json')):
    s = json.load(open(f))
    print(f"{s['dataset']:<26} {s['model']:<10} "
          f"WER {s['wer_norm_pct']:>6.2f}%  CER {s['cer_norm_pct']:>6.2f}%  "
          f"n={s['utterances']}")

In [ ]:
# Only needed if WHISPER_REPRO_DATA was local rather than Drive
# !rsync -a /content/whisper-data/ /content/drive/MyDrive/whisper-repro-data/